# Flatfox Scraper for Central Switzerland

This notebook collects all listings published on [flatfox.ch](https://flatfox.ch) in Central Switzerland (Lucerne, Uri, Schwyz, Obwalden, Nidwalden, Zug) and stores them as raw data.

## Approach

1. **Define the search area:** `BOUNDS` is a rectangle in WGS 84 coordinates that covers all of Central Switzerland. The corner points were determined on [map.geo.admin.ch](https://map.geo.admin.ch). The rectangle also includes areas outside the six cantons. These listings are filtered out later in the `data_engineering_real_estate` notebook.
2. **Find listings on the map:** The map API (`/api/v1/pin/`) returns at most `MAX_COUNT` listings per request. If this limit is reached, the notebook splits the rectangle into four sub-rectangles and queries each one separately. This repeats down to a depth of `MAX_DEPTH`, until every sub-area is fully covered. Duplicate hits are merged by their listing ID (`pk`).
3. **Fetch details:** For all IDs found, the notebook loads the full listing data (`/api/v1/public-listing/`) in batches of `BATCH_SIZE` listings.
4. **Build the table:** The details are flattened into a table. Each reported amenity (e.g. balcony, lift, garage) becomes its own yes/no column `attribute_reported_<name>`.

The notebook waits `REQUEST_PAUSE` seconds between requests to avoid overloading the server.

## Output

Each run gets its own timestamped folder under `data/flatfox_data/`:

| File | Content |
|---|---|
| `run_config.json` | Settings used for the run |
| `pins_tile_XXXX.json` | Raw response of each map request |
| `pins_raw.json` | All listings found and the search log |
| `details_batch_XXXX.json` | Raw response of each detail request |
| `listings_raw.json` | All listing details with check information (missing IDs, unresolved areas) |
| `listings_all_fields.csv` | Table with all fields, used as input for further processing |

The results of several runs are then merged and cleaned in the `data_engineering_real_estate` notebook.

## Search area

The search area is a rectangle that covers all six Central Swiss cantons. Each side of the rectangle is set by the outermost point of a canton in that direction. The points were read off [map.geo.admin.ch](https://map.geo.admin.ch) in WGS 84 coordinates (latitude, longitude).

| Bound | Outermost point | Latitude | Longitude | Value used |
|---|---|---|---|---|
| `west` | Westernmost point of the canton of Lucerne | 47.227713 | 7.838717 | 7.838717 |
| `east` | Easternmost point of the canton of Schwyz | 47.170131 | 9.005109 | 9.005109 |
| `south` | Southernmost point of the canton of Uri | 46.526306 | 8.478879 | 46.526306 |
| `north` | Northernmost point of the canton of Lucerne | 47.285069 | 8.257735 | 47.285069 |

`west` and `east` use the longitude of their point, `south` and `north` use the latitude. Together they define the bottom-left corner (south/west) and the top-right corner (north/east) of the rectangle.

Because the cantons are not rectangular, the search area also covers parts of neighbouring cantons. These listings are removed later in the `data_engineering_real_estate` notebook.

In [ ]:
#%pip install requests pandas

In [ ]:
import json
import time
import pandas as pd
import requests
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

In [ ]:
BASE_URL = "https://flatfox.ch"

BOUNDS = {
    "west": 7.838717,
    "east": 9.005109,
    "south": 46.526306,
    "north": 47.285069,
}

MAX_COUNT = 400
BATCH_SIZE = 50

REQUEST_TIMEOUT = 30
REQUEST_PAUSE = 2
MAX_DEPTH = 8
MAX_SEARCH_REQUESTS = 1000

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
OUTPUT_ROOT = PROJECT_DIR / "data" / "flatfox_data"

In [ ]:
# Helper functions
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, data):
    """Save data as UTF-8 JSON."""
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

def get_list(session, path, params):
    """Send an API request and expect a list of listings."""
    time.sleep(REQUEST_PAUSE)

    response = session.get(
        urljoin(BASE_URL, path),
        params=params,
        timeout=REQUEST_TIMEOUT,
    )

    if not response.ok:
        raise requests.HTTPError(
            f"HTTP {response.status_code} bei {response.url}\n"
            f"Response excerpt: {response.text[:1000]}",
            response=response,
        )

    data = response.json()

    if not isinstance(data, list):
        raise ValueError(
            f"{path}: Unexpected response structure. "
            "Expected a JSON list."
        )

    if any(
        not isinstance(item, dict) or "pk" not in item
        for item in data
    ):
        raise ValueError(
            f"{path}: At least one entry has no listing ID."
        )
    return data

def split_bounds(bounds):
    """Split a rectangle into four sub-rectangles."""
    middle_longitude = (bounds["west"] + bounds["east"]) / 2
    middle_latitude = (bounds["south"] + bounds["north"]) / 2

    return [
        {
            "west": west,
            "east": east,
            "south": south,
            "north": north,
        }
        for west, east in [
            (bounds["west"], middle_longitude),
            (middle_longitude, bounds["east"]),
        ]
        for south, north in [
            (bounds["south"], middle_latitude),
            (middle_latitude, bounds["north"]),
        ]
    ]

def attribute_names(record):
    """Read the reported amenity codes of a listing."""
    attributes = record.get("attributes")

    if not isinstance(attributes, list):
        return None

    return {
        item["name"]
        for item in attributes
        if isinstance(item, dict)
        and isinstance(item.get("name"), str)
    }

def csv_value(value):
    """Keep lists and dictionaries intact inside CSV cells."""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)

    return value

In [ ]:
def main():
    if not (-180 <= BOUNDS["west"] < BOUNDS["east"] <= 180 and
            -90 <= BOUNDS["south"] < BOUNDS["north"] <= 90):
        
        raise ValueError("Invalid limits in BOUNDS.")

    started_at = datetime.now(timezone.utc)

    # Separate output folder for each run.
    run_name = started_at.strftime("%Y%m%dT%H%M%S_%fZ")
    output_dir = OUTPUT_ROOT / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    save_json(
        output_dir / "run_config.json",
        {
            "started_at": started_at.isoformat(),
            "bounds": BOUNDS,
            "max_count": MAX_COUNT,
            "batch_size": BATCH_SIZE,
            "max_depth": MAX_DEPTH,
            "max_search_requests": MAX_SEARCH_REQUESTS,
            "request_pause": REQUEST_PAUSE,
        },
    )

    with requests.Session() as session:

        # Search for pins and split search areas that hit the limit
        print("Searching for listings in the defined map area ...")

        pending = [(BOUNDS.copy(), 0)]
        pins_by_id = {}
        search_log = []
        unresolved_areas = []
        request_number = 0

        while pending:
            if request_number >= MAX_SEARCH_REQUESTS:
                unresolved_areas.extend(
                    {
                        "bounds": bounds,
                        "depth": depth,
                        "reason": "request_budget_exhausted",
                    }
                    for bounds, depth in pending
                )
                break

            bounds, depth = pending.pop()

            tile_pins = get_list(
                session,
                "/api/v1/pin/",
                {
                    **bounds,
                    "max_count": MAX_COUNT,
                },
            )

            request_number += 1
            limit_reached = len(tile_pins) >= MAX_COUNT

            tile_metadata = {
                "fetched_at": utc_now(),
                "bounds": bounds,
                "depth": depth,
                "returned_count": len(tile_pins),
                "limit_reached": limit_reached,
            }

            # Save every map response in full
            save_json(
                output_dir / f"pins_tile_{request_number:04d}.json",
                {
                    **tile_metadata,
                    "pins": tile_pins,
                },
            )

            search_log.append(tile_metadata)

            # Also keep hits from parent rectangles.
            # Repeated IDs are merged here.
            for pin in tile_pins:
                pins_by_id[pin["pk"]] = pin

            print(
                f"Search request {request_number}: "
                f"depth {depth}, "
                f"{len(tile_pins)} pins, "
                f"{len(pins_by_id)} unique IDs in total"
            )

            if limit_reached:
                if depth >= MAX_DEPTH:
                    unresolved_areas.append({
                        "bounds": bounds,
                        "depth": depth,
                        "reason": "limit_reached_at_max_depth",
                    })
                else:
                    pending.extend(
                        (child, depth + 1)
                        for child in split_bounds(bounds)
                    )

        pins = list(pins_by_id.values())
        ids = sorted(pins_by_id)

        save_json(
            output_dir / "pins_raw.json",
            {
                "started_at": started_at.isoformat(),
                "search_finished_at": utc_now(),
                "search_bounds": BOUNDS,
                "max_count": MAX_COUNT,
                "request_count": request_number,
                "unresolved_areas": unresolved_areas,
                "search_log": search_log,
                "pins": pins,
            },
        )

        print(f"\nUnique listing IDs: {len(ids)}")

        if unresolved_areas:
            print(f"WARNING: {len(unresolved_areas)} sub-areas have not been fully processed. Their bounds are stored in pins_raw.json. Listings found so far are still processed.")

        if not ids:
            print(f"No listings found. Output: {output_dir.resolve()}")
            return pd.DataFrame()

        # Fetch full listing details
        listings = []
        batch_metadata = []
        total_batches = (len(ids) + BATCH_SIZE - 1) // BATCH_SIZE

        for batch_number, start in enumerate(
            range(0, len(ids), BATCH_SIZE),
            start=1,
        ):
            batch_ids = ids[start:start + BATCH_SIZE]

            params = [
                ("limit", "0"),
                ("expand", "cover_image"),
            ] + [
                ("pk", str(listing_id))
                for listing_id in batch_ids
            ]

            print(f"Loading detail batch {batch_number}/{total_batches} with {len(batch_ids)} IDs ...")

            batch_data = get_list(session,"/api/v1/public-listing/", params,)

            fetched_at = utc_now()

            # Save every detail response immediately.
            save_json(
                output_dir / f"details_batch_{batch_number:04d}.json",
                {
                    "fetched_at": fetched_at,
                    "requested_ids": batch_ids,
                    "listings": batch_data,
                },
            )

            listings.extend(batch_data)

            batch_metadata.append({
                "batch": batch_number,
                "fetched_at": fetched_at,
                "requested_ids": batch_ids,
                "returned_count": len(batch_data),
            })

    # Save the complete raw data
    received_ids = {
        listing["pk"]
        for listing in listings
    }

    missing_ids = sorted(set(ids) - received_ids)
    unexpected_ids = sorted(received_ids - set(ids))

    save_json(
        output_dir / "listings_raw.json",
        {
            "started_at": started_at.isoformat(),
            "finished_at": utc_now(),
            "search_bounds": BOUNDS,
            "requested_ids": ids,
            "missing_ids": missing_ids,
            "unexpected_ids": unexpected_ids,
            "unresolved_areas": unresolved_areas,
            "batches": batch_metadata,
            "listings": listings,
        },
    )

    if not listings:
        print("No listing details received.")
        print(f"Raw data saved: {output_dir.resolve()}")
        return pd.DataFrame()

    # Flatten all detail fields into a table
    df = pd.json_normalize(listings, sep=".")

    df["collection_started_at"] = started_at.isoformat()

    df["source_url"] = [
        urljoin(BASE_URL, record["url"])
        if isinstance(record.get("url"), str)
        else None
        for record in listings
    ]

    # Add amenities as additional columns
    attribute_sets = [attribute_names(record) for record in listings]

    all_attributes = sorted({name for names in attribute_sets if names is not None for name in names})

    for name in all_attributes:
        df[f"attribute_reported_{name}"] = pd.array(
            [
                name in names
                if names is not None
                else pd.NA
                for names in attribute_sets
            ],
            dtype="boolean",
        )

    # Save CSV with all fields
    csv_df = df.copy()

    for column in csv_df.columns:
        csv_df[column] = csv_df[column].map(csv_value)

    csv_path = output_dir / "listings_all_fields.csv"
    
    csv_df.to_csv(csv_path, index=False, encoding="utf-8-sig",)

    print("\nData collection completed.")
    print(f"Map requests: {request_number}")
    print(f"Unique IDs found: {len(ids)}")
    print(f"Detail records: {len(df)}")
    print(f"Unique listings with details: {len(received_ids)}")
    print(f"Table columns: {len(df.columns)}")
    print(f"IDs not returned: {missing_ids}")
    print(f"Unexpectedly returned IDs: {unexpected_ids}")
    print(f"Unresolved sub-areas: {len(unresolved_areas)}")
    print(f"Amenity codes found: {all_attributes}")
    print(f"Output folder: {output_dir.resolve()}")

    return df

df = main()